In [39]:
pip install numpy pandas matplotlib seaborn scikit-learn xgboost lightgbm


Note: you may need to restart the kernel to use updated packages.


In [40]:
import tensorflow as tf
tf.config.list_physical_devices('GPU')


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [41]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


TensorFlow: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [42]:
DATA_DIR = "D:\M5"

calendar_path = f"{DATA_DIR}/calendar.csv"
sales_path = f"{DATA_DIR}/sales_train_validation.csv"
prices_path = f"{DATA_DIR}/sell_prices.csv"

calendar_path, sales_path, prices_path


('D:\\M5/calendar.csv',
 'D:\\M5/sales_train_validation.csv',
 'D:\\M5/sell_prices.csv')

In [43]:
import pandas as pd

calendar = pd.read_csv("calendar.csv")

print("Calendar shape:", calendar.shape)
calendar.head()


Calendar shape: (1969, 14)


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [44]:
sell_prices = pd.read_csv("sell_prices.csv")

print("Sell prices shape:", sell_prices.shape)
sell_prices.head()


Sell prices shape: (6841121, 4)


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [45]:
# Load only the first 6 columns (metadata), NOT the 1900+ day columns
sales_meta = pd.read_csv(
    "sales_train_validation.csv",
    usecols=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
)

print("Sales metadata shape:", sales_meta.shape)
sales_meta.head()


Sales metadata shape: (30490, 6)


,id,item_id,dept_id,cat_id,store_id,state_id
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA


In [46]:
# Read only column names to inspect day range (NO data loaded)
sales_cols = pd.read_csv(
    "sales_train_validation.csv",
    nrows=0
).columns.tolist()

print("Total columns:", len(sales_cols))
print("First 10 columns:", sales_cols[:10])
print("Last 10 columns:", sales_cols[-10:])


Total columns: 1919
First 10 columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd_1', 'd_2', 'd_3', 'd_4']
Last 10 columns: ['d_1904', 'd_1905', 'd_1906', 'd_1907', 'd_1908', 'd_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


In [47]:
import pandas as pd
from collections import defaultdict

DATA_PATH = "sales_train_validation.csv"

# Dictionary to store total sales per SKU id
sku_sales = defaultdict(int)

# Read file in chunks to avoid RAM crash
chunk_size = 1000  # safe
for chunk in pd.read_csv(DATA_PATH, chunksize=chunk_size):
    day_cols = [c for c in chunk.columns if c.startswith("d_")]
    
    # Sum sales across all days for each SKU row
    chunk["total_sales"] = chunk[day_cols].sum(axis=1)
    
    for sku, total in zip(chunk["id"], chunk["total_sales"]):
        sku_sales[sku] += total

# Convert to DataFrame
sku_sales_df = pd.DataFrame(
    sku_sales.items(), columns=["sku_id", "total_units_sold"]
)

# Select top 20 selling SKUs
top_20_skus = (
    sku_sales_df
    .sort_values("total_units_sold", ascending=False)
    .head(300)
    .reset_index(drop=True)
)

print("Top 20 SKUs selected:")
top_20_skus


Top 20 SKUs selected:


,sku_id,total_units_sold
0,FOODS_3_090_CA_3_validation,250502
1,FOODS_3_586_TX_2_validation,192835
2,FOODS_3_586_TX_3_validation,150122
3,FOODS_3_586_CA_3_validation,134386
4,FOODS_3_090_CA_1_validation,127203
...,...,...
295,FOODS_3_234_TX_1_validation,21484
296,HOUSEHOLD_1_151_CA_3_validation,21472
297,FOODS_3_362_CA_1_validation,21426
298,FOODS_3_109_WI_2_validation,21282


In [48]:
# Create inventory name mapping from SKU id

def make_inventory_name(sku_id):
    parts = sku_id.split("_")
    category = parts[0]          # FOODS
    dept = parts[1]              # 3
    item = parts[2]              # 090
    store = parts[3] + "_" + parts[4]  # CA_3
    return f"{category} | Dept {dept} | Item {item} | Store {store}"

top_20_skus["inventory_name"] = top_20_skus["sku_id"].apply(make_inventory_name)

top_20_skus


,sku_id,total_units_sold,inventory_name
0,FOODS_3_090_CA_3_validation,250502,FOODS | Dept 3 | Item 090 | Store CA_3
1,FOODS_3_586_TX_2_validation,192835,FOODS | Dept 3 | Item 586 | Store TX_2
2,FOODS_3_586_TX_3_validation,150122,FOODS | Dept 3 | Item 586 | Store TX_3
3,FOODS_3_586_CA_3_validation,134386,FOODS | Dept 3 | Item 586 | Store CA_3
4,FOODS_3_090_CA_1_validation,127203,FOODS | Dept 3 | Item 090 | Store CA_1
...,...,...,...
295,FOODS_3_234_TX_1_validation,21484,FOODS | Dept 3 | Item 234 | Store TX_1
296,HOUSEHOLD_1_151_CA_3_validation,21472,HOUSEHOLD | Dept 1 | Item 151 | Store CA_3
297,FOODS_3_362_CA_1_validation,21426,FOODS | Dept 3 | Item 362 | Store CA_1
298,FOODS_3_109_WI_2_validation,21282,FOODS | Dept 3 | Item 109 | Store WI_2


In [49]:
import pandas as pd

DATA_PATH = "sales_train_validation.csv"
top_sku_ids = set(top_20_skus["sku_id"])

sales_long_list = []

chunk_size = 500  # safe for RAM
for chunk in pd.read_csv(DATA_PATH, chunksize=chunk_size):
    # Filter only top-20 SKUs
    chunk = chunk[chunk["id"].isin(top_sku_ids)]
    if chunk.empty:
        continue

    # Identify day columns
    day_cols = [c for c in chunk.columns if c.startswith("d_")]

    # Convert wide to long
    melted = chunk.melt(
        id_vars=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"],
        value_vars=day_cols,
        var_name="d",
        value_name="sales"
    )

    sales_long_list.append(melted)

# Combine all chunks
sales_long = pd.concat(sales_long_list, ignore_index=True)

print("Sales long format shape:", sales_long.shape)
sales_long.head()


Sales long format shape: (573900, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
0,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_1,9
1,HOBBIES_1_371_CA_1_validation,HOBBIES_1_371,HOBBIES_1,HOBBIES,CA_1,CA,d_1,14
2,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_2,24
3,HOBBIES_1_371_CA_1_validation,HOBBIES_1_371,HOBBIES_1,HOBBIES,CA_1,CA,d_2,25
4,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0


In [50]:
# Merge sales data with calendar on day key `d`
sales_calendar = sales_long.merge(
    calendar,
    on="d",
    how="left"
)

print("After calendar merge shape:", sales_calendar.shape)
sales_calendar.head()


After calendar merge shape: (573900, 21)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_1,9,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
1,HOBBIES_1_371_CA_1_validation,HOBBIES_1_371,HOBBIES_1,HOBBIES,CA_1,CA,d_1,14,2011-01-29,11101,...,1,1,2011,NaN,NaN,NaN,NaN,0,0,0
2,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_2,24,2011-01-30,11101,...,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
3,HOBBIES_1_371_CA_1_validation,HOBBIES_1_371,HOBBIES_1,HOBBIES,CA_1,CA,d_2,25,2011-01-30,11101,...,2,1,2011,NaN,NaN,NaN,NaN,0,0,0
4,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0,2011-01-31,11101,...,3,1,2011,NaN,NaN,NaN,NaN,0,0,0


In [51]:
# Merge sales+calendar with sell prices
sales_full = sales_calendar.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

print("After price merge shape:", sales_full.shape)

# Check missing prices
print("Missing sell_price count:", sales_full["sell_price"].isna().sum())

sales_full.head()


After price merge shape: (573900, 22)
Missing sell_price count: 23541


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_1,9,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.46
1,HOBBIES_1_371_CA_1_validation,HOBBIES_1_371,HOBBIES_1,HOBBIES,CA_1,CA,d_1,14,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.46
2,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_2,24,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.46
3,HOBBIES_1_371_CA_1_validation,HOBBIES_1_371,HOBBIES_1,HOBBIES,CA_1,CA,d_2,25,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.46
4,HOBBIES_1_348_CA_1_validation,HOBBIES_1_348,HOBBIES_1,HOBBIES,CA_1,CA,d_3,0,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.46


In [52]:
# Sort data by SKU and date
sales_full = sales_full.sort_values(
    by=["id", "date"]
).reset_index(drop=True)

# Forward-fill missing prices per SKU
sales_full["sell_price"] = (
    sales_full
    .groupby("id")["sell_price"]
    .ffill()
    .bfill()
)

print("Remaining missing prices:", sales_full["sell_price"].isna().sum())

sales_full.head()


Remaining missing prices: 0


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_1,12,2011-01-29,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.92
1,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_2,4,2011-01-30,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.92
2,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_3,6,2011-01-31,11101,...,1,2011,NaN,NaN,NaN,NaN,0,0,0,0.92
3,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_4,0,2011-02-01,11101,...,2,2011,NaN,NaN,NaN,NaN,1,1,0,0.92
4,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_5,3,2011-02-02,11101,...,2,2011,NaN,NaN,NaN,NaN,1,0,1,0.92


In [53]:
# Create lag features per SKU
for lag in [1, 7, 14]:
    sales_full[f"lag_{lag}"] = (
        sales_full
        .groupby("id")["sales"]
        .shift(lag)
    )

# Rolling mean features
sales_full["rmean_7"] = (
    sales_full
    .groupby("id")["sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

sales_full["rmean_14"] = (
    sales_full
    .groupby("id")["sales"]
    .shift(1)
    .rolling(14)
    .mean()
)

sales_full.head(10)


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,...,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,lag_1,lag_7,lag_14,rmean_7,rmean_14
0,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_1,12,2011-01-29,11101,...,NaN,0,0,0,0.92,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_2,4,2011-01-30,11101,...,NaN,0,0,0,0.92,12.0,NaN,NaN,NaN,NaN
2,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_3,6,2011-01-31,11101,...,NaN,0,0,0,0.92,4.0,NaN,NaN,NaN,NaN
3,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_4,0,2011-02-01,11101,...,NaN,1,1,0,0.92,6.0,NaN,NaN,NaN,NaN
4,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_5,3,2011-02-02,11101,...,NaN,1,0,1,0.92,0.0,NaN,NaN,NaN,NaN
5,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_6,10,2011-02-03,11101,...,NaN,1,1,1,0.92,3.0,NaN,NaN,NaN,NaN
6,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_7,13,2011-02-04,11101,...,NaN,1,0,0,0.92,10.0,NaN,NaN,NaN,NaN
7,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_8,1,2011-02-05,11102,...,NaN,1,1,1,0.92,13.0,12.0,NaN,6.857143,NaN
8,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_9,6,2011-02-06,11102,...,NaN,1,1,1,0.92,1.0,4.0,NaN,5.285714,NaN
9,FOODS_1_046_TX_2_validation,FOODS_1_046,FOODS_1,FOODS,TX_2,TX,d_10,2,2011-02-07,11102,...,NaN,1,1,0,0.92,6.0,6.0,NaN,5.571429,NaN


In [54]:
# Drop rows with NaNs caused by lag features
train_df = sales_full.dropna().reset_index(drop=True)

# Target variable
y = train_df["sales"]

# Feature columns
feature_cols = [
    "lag_1", "lag_7", "lag_14",
    "rmean_7", "rmean_14",
    "sell_price",
    "snap_CA", "snap_TX", "snap_WI",
    "wday", "month"
]

X = train_df[feature_cols]

print("Training shape:", X.shape)
X.head()


Training shape: (1200, 11)


,lag_1,lag_7,lag_14,rmean_7,rmean_14,sell_price,snap_CA,snap_TX,snap_WI,wday,month
0,5.0,4.0,2.0,3.142857,3.142857,0.92,0,0,0,2,4
1,3.0,15.0,20.0,7.000000,8.857143,0.98,1,1,1,2,5
2,27.0,22.0,34.0,20.857143,19.285714,0.98,0,0,0,2,4
3,11.0,9.0,19.0,11.000000,11.857143,0.98,0,1,1,2,6
4,1.0,8.0,5.0,1.428571,3.071429,1.00,0,0,0,2,4


In [55]:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    from xgboost import XGBRegressor
    import numpy as np

    # Explicit numeric feature list (VERY IMPORTANT)
    feature_cols = [
        "lag_1", "lag_7", "lag_14",
        "rmean_7", "rmean_14",
        "sell_price",
        "snap_CA", "snap_TX", "snap_WI",
        "wday", "month"
    ]

    # Use ONLY numeric features
    X = train_df[feature_cols]
    y = train_df["sales"]

    # Time-based split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    # XGBoost baseline
    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    )

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_val)


    # Metrics
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)


    print(f"RMSE: {rmse:.2f}")
    print(f"MAE   : {mae:.2f}")


RMSE: 12.26
MAE   : 8.28


In [56]:
# ==============================
# FINAL LSTM SEQUENCE PREPARATION
# ==============================

import numpy as np

SEQ_LEN = 28  # longer context → stronger than XGBoost

feature_cols = [
    "lag_1", "lag_7", "lag_14",
    "rmean_7", "rmean_14",
    "sell_price",
    "snap_CA", "snap_TX", "snap_WI",
    "wday", "month"
]

# sales_full already exists from previous pipeline
data = sales_full[feature_cols + ["sales"]].dropna().reset_index(drop=True)

X_seq, y_seq = [], []

for i in range(SEQ_LEN, len(data)):
    X_seq.append(data[feature_cols].iloc[i-SEQ_LEN:i].values)
    y_seq.append(data["sales"].iloc[i])

X_seq = np.array(X_seq, dtype=np.float32)
y_seq = np.array(y_seq, dtype=np.float32)

# Time-based split (NO SHUFFLE)
split_idx = int(0.8 * len(X_seq))
X_train, X_val = X_seq[:split_idx], X_seq[split_idx:]
y_train, y_val = y_seq[:split_idx], y_seq[split_idx:]

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)


Train shape: (455737, 28, 11)
Validation shape: (113935, 28, 11)


In [57]:
# ==========================================================
# FINAL LSTM + SELF-ATTENTION MODEL (SINGLE CLEAN CELL)
# ==========================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dropout, Dense,
    MultiHeadAttention, LayerNormalization,
    GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -------------------------------
# 1. MODEL ARCHITECTURE
# -------------------------------
SEQ_LEN = X_train.shape[1]
N_FEATURES = X_train.shape[2]

inputs = Input(shape=(SEQ_LEN, N_FEATURES))

x = LSTM(64, return_sequences=True)(inputs)
x = Dropout(0.2)(x)

attn = MultiHeadAttention(
    num_heads=4,
    key_dim=16
)(x, x)

x = x + attn
x = LayerNormalization()(x)

x = GlobalAveragePooling1D()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.2)(x)

outputs = Dense(1)(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

model.summary()

# -------------------------------
# 2. CALLBACKS
# -------------------------------
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "final_lstm_attention_model.h5",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

# -------------------------------
# 3. TRAINING (WITH PROGRESS BAR)
# -------------------------------
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# -------------------------------
# 4. PREDICTION
# -------------------------------
y_pred = model.predict(X_val).flatten()
y_true = y_val.flatten()

# -------------------------------
# 5. METRICS (RETAIL-SAFE)
# -------------------------------
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)

# Safe MAPE (ignore zero sales)
mask = y_true > 0
safe_mape = np.mean(
    np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
) * 100

# sMAPE (preferred for retail)
smape = np.mean(
    2 * np.abs(y_pred - y_true) /
    (np.abs(y_true) + np.abs(y_pred) + 1e-6)
) * 100

# -------------------------------
# 6. FINAL OUTPUT
# -------------------------------
print("\n===== FINAL LSTM + ATTENTION PERFORMANCE =====")
print(f"RMSE  : {rmse:.2f}")
print(f"MAE   : {mae:.2f}")


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 28, 11)]     0           []                               
                                                                                                  
 lstm (LSTM)                    (None, 28, 64)       19456       ['input_1[0][0]']                
                                                                                                  
 dropout (Dropout)              (None, 28, 64)       0           ['lstm[0][0]']                   
                                                                                                  
 multi_head_attention (MultiHea  (None, 28, 64)      16640       ['dropout[0][0]',                
 dAttention)                                                      'dropout[0][0]']            

In [58]:
# Create SKU → Inventory name mapping
sku_inventory_map = dict(
    zip(top_20_skus["sku_id"], top_20_skus["inventory_name"])
)

print("SKU inventory map created successfully")


SKU inventory map created successfully


In [59]:
# =====================================================
# FINAL SKU-WISE FORECASTING (7 & 14 DAYS)
# =====================================================

import numpy as np
import pandas as pd
import tensorflow as tf

# Load trained model
model = tf.keras.models.load_model("final_lstm_attention_model.h5")

SEQ_LEN = 28
FORECAST_7 = 7
FORECAST_14 = 14

results = []

# Loop through TOP 20 SKUs
for sku in top_20_skus["sku_id"]:

    sku_data = sales_full[sales_full["id"] == sku].copy()
    sku_data = sku_data.sort_values("date")

    feature_cols = [
        "lag_1", "lag_7", "lag_14",
        "rmean_7", "rmean_14",
        "sell_price",
        "snap_CA", "snap_TX", "snap_WI",
        "wday", "month"
    ]

    sku_data = sku_data[feature_cols + ["sales"]].dropna()

    if len(sku_data) < SEQ_LEN:
        continue

    last_seq = sku_data[feature_cols].iloc[-SEQ_LEN:].values
    current_seq = last_seq.copy()

    preds_7, preds_14 = [], []

    for day in range(FORECAST_14):
        pred = model.predict(current_seq[np.newaxis, ...], verbose=0)[0][0]
        pred = max(0, pred)  # no negative sales

        if day < FORECAST_7:
            preds_7.append(pred)
        preds_14.append(pred)

        # Update rolling window
        new_row = current_seq[-1].copy()
        new_row[0] = pred
        current_seq = np.vstack([current_seq[1:], new_row])

    results.append({
        "sku_id": sku,
        "inventory_name": sku_inventory_map[sku],
        "7_day_units": int(np.round(sum(preds_7))),
        "14_day_units": int(np.round(sum(preds_14)))
    })

forecast_df = pd.DataFrame(results)

# Overall requirement
overall_7 = forecast_df["7_day_units"].sum()
overall_14 = forecast_df["14_day_units"].sum()

print("\n===== OVERALL INVENTORY REQUIREMENT =====")
print(f"Total units to purchase (7 days) : {overall_7}")
print(f"Total units to purchase (14 days): {overall_14}")

print("\n===== SKU-WISE FORECAST =====")
forecast_df



===== OVERALL INVENTORY REQUIREMENT =====
Total units to purchase (7 days) : 37564
Total units to purchase (14 days): 73829

===== SKU-WISE FORECAST =====


,sku_id,inventory_name,7_day_units,14_day_units
0,FOODS_3_090_CA_3_validation,FOODS | Dept 3 | Item 090 | Store CA_3,704,1278
1,FOODS_3_586_TX_2_validation,FOODS | Dept 3 | Item 586 | Store TX_2,457,870
2,FOODS_3_586_TX_3_validation,FOODS | Dept 3 | Item 586 | Store TX_3,378,737
3,FOODS_3_586_CA_3_validation,FOODS | Dept 3 | Item 586 | Store CA_3,435,840
4,FOODS_3_090_CA_1_validation,FOODS | Dept 3 | Item 090 | Store CA_1,326,629
...,...,...,...,...
295,FOODS_3_234_TX_1_validation,FOODS | Dept 3 | Item 234 | Store TX_1,98,180
296,HOUSEHOLD_1_151_CA_3_validation,HOUSEHOLD | Dept 1 | Item 151 | Store CA_3,6,13
297,FOODS_3_362_CA_1_validation,FOODS | Dept 3 | Item 362 | Store CA_1,86,146
298,FOODS_3_109_WI_2_validation,FOODS | Dept 3 | Item 109 | Store WI_2,86,198


In [60]:
# ==============================
# XGBOOST 7 & 14 DAY FORECAST
# ==============================

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

# ------------------------------
# Configuration
# ------------------------------
FORECAST_DAYS_7 = 7
FORECAST_DAYS_14 = 14

feature_cols = [
    "lag_1", "lag_7", "lag_14",
    "rmean_7", "rmean_14",
    "sell_price",
    "snap_CA", "snap_TX", "snap_WI",
    "wday", "month"
]

# ------------------------------
# Train single global XGBoost
# ------------------------------
train_df = sales_full[feature_cols + ["sales"]].dropna().reset_index(drop=True)

X_train = train_df[feature_cols]
y_train = train_df["sales"]

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("✅ XGBoost model trained successfully")

# ------------------------------
# Forecast per SKU
# ------------------------------
results = []

for sku in top_20_skus["sku_id"]:
    sku_df = sales_full[sales_full["id"] == sku].copy()
    sku_df = sku_df.sort_values("date")

    # Take last valid row
    current_row = sku_df.iloc[-1].copy()
    preds_7, preds_14 = [], []

    for day in range(FORECAST_DAYS_14):
        X_input = current_row[feature_cols].values.reshape(1, -1)
        pred = xgb_model.predict(X_input)[0]

        if day < FORECAST_DAYS_7:
            preds_7.append(pred)
        preds_14.append(pred)

        # Update lags
        current_row["lag_14"] = current_row["lag_7"]
        current_row["lag_7"] = current_row["lag_1"]
        current_row["lag_1"] = pred

        # Update rolling means
        current_row["rmean_7"] = np.mean([current_row["lag_1"], current_row["lag_7"]])
        current_row["rmean_14"] = np.mean([current_row["lag_1"], current_row["lag_14"]])

        # Advance calendar features
        current_row["wday"] = (current_row["wday"] % 7) + 1

    results.append({
        "sku_id": sku,
        "inventory_name": sku_inventory_map[sku],
        "xgb_7_day_units": int(np.round(np.sum(preds_7))),
        "xgb_14_day_units": int(np.round(np.sum(preds_14)))
    })

# ------------------------------
# Final Forecast Tables
# ------------------------------
xgb_forecast_df = pd.DataFrame(results)

overall_7 = xgb_forecast_df["xgb_7_day_units"].sum()
overall_14 = xgb_forecast_df["xgb_14_day_units"].sum()

print("\n===== XGBOOST OVERALL INVENTORY REQUIREMENT =====")
print(f"Total units (7 days) : {overall_7}")
print(f"Total units (14 days): {overall_14}")

print("\n===== XGBOOST SKU-WISE FORECAST =====")
display(xgb_forecast_df)


✅ XGBoost model trained successfully

===== XGBOOST OVERALL INVENTORY REQUIREMENT =====
Total units (7 days) : 39346
Total units (14 days): 72260

===== XGBOOST SKU-WISE FORECAST =====


,sku_id,inventory_name,xgb_7_day_units,xgb_14_day_units
0,FOODS_3_090_CA_3_validation,FOODS | Dept 3 | Item 090 | Store CA_3,736,1452
1,FOODS_3_586_TX_2_validation,FOODS | Dept 3 | Item 586 | Store TX_2,578,1072
2,FOODS_3_586_TX_3_validation,FOODS | Dept 3 | Item 586 | Store TX_3,394,727
3,FOODS_3_586_CA_3_validation,FOODS | Dept 3 | Item 586 | Store CA_3,471,886
4,FOODS_3_090_CA_1_validation,FOODS | Dept 3 | Item 090 | Store CA_1,482,908
...,...,...,...,...
295,FOODS_3_234_TX_1_validation,FOODS | Dept 3 | Item 234 | Store TX_1,82,153
296,HOUSEHOLD_1_151_CA_3_validation,HOUSEHOLD | Dept 1 | Item 151 | Store CA_3,7,63
297,FOODS_3_362_CA_1_validation,FOODS | Dept 3 | Item 362 | Store CA_1,119,213
298,FOODS_3_109_WI_2_validation,FOODS | Dept 3 | Item 109 | Store WI_2,91,166


In [61]:
# =========================================
# DASHBOARD STEP 1: FINAL MERGED DATAFRAME
# =========================================

import pandas as pd

# -----------------------------
# 1. Sanity check
# -----------------------------
required_dfs = ["xgb_forecast_df", "forecast_df"]
missing = [df for df in required_dfs if df not in globals()]

if missing:
    raise ValueError(f"Missing required dataframe(s): {missing}")

# Rename for clarity
lstm_forecast_df = forecast_df.copy()

# -----------------------------
# 2. Standardize column names
# -----------------------------
xgb_forecast_df = xgb_forecast_df.rename(columns={
    "xgb_7_day_units": "xgb_7_day",
    "xgb_14_day_units": "xgb_14_day"
})

lstm_forecast_df = lstm_forecast_df.rename(columns={
    "7_day_units": "lstm_7_day",
    "14_day_units": "lstm_14_day"
})

# -----------------------------
# 3. Merge XGBoost + LSTM
# -----------------------------
dashboard_df = pd.merge(
    xgb_forecast_df,
    lstm_forecast_df[
        ["sku_id", "lstm_7_day", "lstm_14_day"]
    ],
    on="sku_id",
    how="inner"
)

# -----------------------------
# 4. Final recommendation logic
# (safe inventory decision)
# -----------------------------
dashboard_df["final_7_day_units"] = (
    dashboard_df[["xgb_7_day", "lstm_7_day"]].mean(axis=1).round().astype(int)
)

dashboard_df["final_14_day_units"] = (
    dashboard_df[["xgb_14_day", "lstm_14_day"]].mean(axis=1).round().astype(int)
)

# -----------------------------
# 5. Reorder columns (dashboard friendly)
# -----------------------------
dashboard_df = dashboard_df[
    [
        "sku_id",
        "inventory_name",
        "xgb_7_day",
        "lstm_7_day",
        "final_7_day_units",
        "xgb_14_day",
        "lstm_14_day",
        "final_14_day_units"
    ]
]

# -----------------------------
# 6. Overall totals (for KPI cards)
# -----------------------------
overall_summary = {
    "Overall 7-Day Units": int(dashboard_df["final_7_day_units"].sum()),
    "Overall 14-Day Units": int(dashboard_df["final_14_day_units"].sum())
}

# -----------------------------
# 7. Output
# -----------------------------
print("✅ DASHBOARD DATA READY\n")
print("===== OVERALL INVENTORY REQUIREMENT =====")
for k, v in overall_summary.items():
    print(f"{k}: {v}")

print("\n===== DASHBOARD DATA PREVIEW =====")
display(dashboard_df.head())


✅ DASHBOARD DATA READY

===== OVERALL INVENTORY REQUIREMENT =====
Overall 7-Day Units: 38459
Overall 14-Day Units: 73039

===== DASHBOARD DATA PREVIEW =====


,sku_id,inventory_name,xgb_7_day,lstm_7_day,final_7_day_units,xgb_14_day,lstm_14_day,final_14_day_units
0,FOODS_3_090_CA_3_validation,FOODS | Dept 3 | Item 090 | Store CA_3,736,704,720,1452,1278,1365
1,FOODS_3_586_TX_2_validation,FOODS | Dept 3 | Item 586 | Store TX_2,578,457,518,1072,870,971
2,FOODS_3_586_TX_3_validation,FOODS | Dept 3 | Item 586 | Store TX_3,394,378,386,727,737,732
3,FOODS_3_586_CA_3_validation,FOODS | Dept 3 | Item 586 | Store CA_3,471,435,453,886,840,863
4,FOODS_3_090_CA_1_validation,FOODS | Dept 3 | Item 090 | Store CA_1,482,326,404,908,629,768


In [62]:
# Save dashboard-ready forecast data
dashboard_df.to_csv(
    "forecast_dashboard_data.csv",
    index=False
)

print("Dashboard data saved successfully")
print("Rows:", dashboard_df.shape[0])
print("Columns:", dashboard_df.shape[1])
dashboard_df.head()


Dashboard data saved successfully
Rows: 300
Columns: 8


,sku_id,inventory_name,xgb_7_day,lstm_7_day,final_7_day_units,xgb_14_day,lstm_14_day,final_14_day_units
0,FOODS_3_090_CA_3_validation,FOODS | Dept 3 | Item 090 | Store CA_3,736,704,720,1452,1278,1365
1,FOODS_3_586_TX_2_validation,FOODS | Dept 3 | Item 586 | Store TX_2,578,457,518,1072,870,971
2,FOODS_3_586_TX_3_validation,FOODS | Dept 3 | Item 586 | Store TX_3,394,378,386,727,737,732
3,FOODS_3_586_CA_3_validation,FOODS | Dept 3 | Item 586 | Store CA_3,471,435,453,886,840,863
4,FOODS_3_090_CA_1_validation,FOODS | Dept 3 | Item 090 | Store CA_1,482,326,404,908,629,768
